In [ ]:
# Install dependencies
!pip install pdfplumber python-docx nltk

# Run
!python ats_resume_analyzer.py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 64.9 MB/s eta 0:00:00
python3: can't open file '/content/ats_resume_analyzer.py': [Errno 2] No such file or directory


In [ ]:
"""
╔══════════════════════════════════════════════════════════════╗
║         ATS Resume Analyzer — Google Colab Edition          ║
║  Works entirely in-browser using ipywidgets + IPython       ║
╚══════════════════════════════════════════════════════════════╝

SETUP (run once in a Colab cell):
    !pip install pdfplumber python-docx nltk ipywidgets -q
    import nltk; nltk.download('stopwords'); nltk.download('wordnet'); nltk.download('punkt')

USAGE:
    Copy this file to Colab, then in a cell run:
        exec(open('ats_resume_analyzer_colab.py').read())
    OR paste everything into a single Colab cell and run it.
"""

# ─── Standard library ─────────────────────────────────────────────────────────
import re
import os
import io

# ─── Colab / IPython display ──────────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ─── Optional NLP / file-parsing libraries ────────────────────────────────────
try:
    import pdfplumber
    HAS_PDFPLUMBER = True
except ImportError:
    HAS_PDFPLUMBER = False

try:
    import PyPDF2
    HAS_PYPDF2 = True
except ImportError:
    HAS_PYPDF2 = False

try:
    from docx import Document as DocxDocument
    HAS_DOCX = True
except ImportError:
    HAS_DOCX = False

try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    STOP_WORDS  = set(stopwords.words("english"))
    LEMMATIZER  = WordNetLemmatizer()
    HAS_NLTK    = True
except Exception:
    HAS_NLTK    = False
    LEMMATIZER  = None
    STOP_WORDS  = {
        "i","me","my","we","our","you","your","he","she","it","its","they","their",
        "what","which","who","this","that","these","those","am","is","are","was","were",
        "be","been","being","have","has","had","do","does","did","will","would","shall",
        "should","may","might","can","could","a","an","the","and","but","or","nor","for",
        "so","yet","both","either","neither","not","only","own","same","than","too","very",
        "just","because","as","until","while","of","at","by","with","about","against",
        "between","into","through","during","before","after","above","below","to","from",
        "in","out","on","off","over","under","again","then","once","here","there","when",
        "where","why","how","all","each","every","more","most","other","some","such","no",
        "any","if","else","than","also","must","s","t","don","doesn","didn","won","isn",
        "aren","wasn","weren","haven","hadn","shouldn","wouldn","couldn","shan",
    }

# ══════════════════════════════════════════════════════════════════════════════
# SKILL TAXONOMY  (100 + skills across domains)
# ══════════════════════════════════════════════════════════════════════════════
SKILL_TAXONOMY = {
    # Languages
    "python","java","javascript","typescript","c++","c#","c","ruby","go","golang",
    "rust","swift","kotlin","scala","r","matlab","perl","php","bash","shell",
    "powershell","vba","dart","lua","haskell","elixir",
    # Web / Frontend
    "html","css","react","angular","vue","svelte","nextjs","nuxtjs","jquery",
    "bootstrap","tailwind","sass","less","webpack","vite","graphql","rest","soap",
    "api","ajax","json","xml",
    # Backend / Frameworks
    "django","flask","fastapi","spring","springboot","express","nodejs","laravel",
    "rails","asp.net","dotnet",".net","hibernate",
    # Databases
    "sql","mysql","postgresql","postgres","sqlite","mongodb","redis","elasticsearch",
    "cassandra","oracle","mssql","dynamodb","firebase","nosql","mariadb","neo4j",
    # Cloud / DevOps
    "aws","azure","gcp","google cloud","docker","kubernetes","k8s","terraform",
    "ansible","jenkins","gitlab","github actions","ci/cd","linux","unix","nginx",
    "apache","heroku","vercel","netlify",
    # Data / ML / AI
    "machine learning","deep learning","nlp","computer vision","ai","tensorflow",
    "pytorch","keras","scikit-learn","sklearn","pandas","numpy","scipy",
    "matplotlib","seaborn","plotly","tableau","power bi","data analysis",
    "data science","data engineering","big data","spark","hadoop","kafka",
    "airflow","dbt","etl","mlops","llm",
    # Tools / Practices
    "git","github","gitlab","bitbucket","jira","confluence","notion","agile",
    "scrum","kanban","devops","tdd","bdd","microservices","oop","solid",
    "design patterns","mvc","rest api",
    # Mobile
    "android","ios","react native","flutter","xamarin","ionic",
    # Security
    "cybersecurity","penetration testing","owasp","ssl","oauth","jwt",
    "encryption","firewall","siem",
    # Soft skills
    "communication","leadership","teamwork","problem solving","critical thinking",
    "project management","time management","collaboration","analytical",
    "presentation","mentoring","documentation",
}

# ══════════════════════════════════════════════════════════════════════════════
# MODULE 1 — TEXT EXTRACTION
# ══════════════════════════════════════════════════════════════════════════════

def extract_text_from_pdf_bytes(file_bytes: bytes) -> str:
    """Extract text from PDF given raw bytes (no temp file needed in Colab)."""
    text = ""
    if HAS_PDFPLUMBER:
        try:
            with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
                for page in pdf.pages:
                    t = page.extract_text()
                    if t:
                        text += t + "\n"
            if text.strip():
                return text
        except Exception:
            pass

    if HAS_PYPDF2:
        try:
            reader = PyPDF2.PdfReader(io.BytesIO(file_bytes))
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    text += t + "\n"
            return text
        except Exception as e:
            raise RuntimeError(f"PyPDF2 error: {e}")

    raise RuntimeError(
        "No PDF library available.\n"
        "Run:  !pip install pdfplumber  then restart runtime."
    )


def extract_text_from_docx_bytes(file_bytes: bytes) -> str:
    """Extract text from DOCX given raw bytes."""
    if not HAS_DOCX:
        raise RuntimeError(
            "python-docx not installed.\nRun:  !pip install python-docx"
        )
    doc = DocxDocument(io.BytesIO(file_bytes))
    return "\n".join(p.text for p in doc.paragraphs if p.text.strip())


def extract_text(filename: str, file_bytes: bytes) -> str:
    """Dispatch extraction by file extension."""
    ext = os.path.splitext(filename)[1].lower()
    if ext == ".pdf":
        return extract_text_from_pdf_bytes(file_bytes)
    elif ext in (".docx", ".doc"):
        return extract_text_from_docx_bytes(file_bytes)
    else:
        raise ValueError(f"Unsupported format '{ext}'. Please upload PDF or DOCX.")


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 2 — PREPROCESSING
# ══════════════════════════════════════════════════════════════════════════════

def preprocess_text(text: str) -> list:
    """Lowercase → clean punctuation → tokenize → stopword filter → lemmatize."""
    text = text.lower()
    text = text.replace("c++","cplusplus").replace("c#","csharp").replace(".net","dotnet")
    text = re.sub(r"[^\w\s']", " ", text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    if HAS_NLTK and LEMMATIZER:
        tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    return tokens


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 3 — SKILL EXTRACTION
# ══════════════════════════════════════════════════════════════════════════════

def extract_skills(raw_text: str) -> set:
    """Match taxonomy skills (single & multi-word) against raw text."""
    text_lower = raw_text.lower()
    found = set()
    for skill in SKILL_TAXONOMY:
        if re.search(r"\b" + re.escape(skill) + r"\b", text_lower):
            found.add(skill)
    return found


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 4 — SCORING
# ══════════════════════════════════════════════════════════════════════════════

def compute_ats_score(resume_skills, jd_skills, resume_tokens, jd_tokens) -> dict:
    """
    Composite ATS score:
      70 % → skill overlap
      30 % → preprocessed keyword overlap
    """
    matched  = resume_skills & jd_skills
    missing  = jd_skills - resume_skills
    extra    = resume_skills - jd_skills

    skill_score = (len(matched) / len(jd_skills) * 100) if jd_skills else 0.0

    rt = set(resume_tokens)
    jt = set(jd_tokens)
    common   = rt & jt
    kw_score = (len(common) / len(jt) * 100) if jt else 0.0

    composite = round(0.70 * skill_score + 0.30 * kw_score, 1)

    return {
        "ats_score":       composite,
        "skill_score":     round(skill_score, 1),
        "keyword_score":   round(kw_score, 1),
        "matched_skills":  sorted(matched),
        "missing_skills":  sorted(missing),
        "extra_skills":    sorted(extra),
        "common_tokens":   len(common),
        "total_jd_tokens": len(jt),
    }


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 5 — SUGGESTIONS
# ══════════════════════════════════════════════════════════════════════════════

def generate_suggestions(result: dict) -> list:
    score   = result["ats_score"]
    missing = result["missing_skills"]
    tips    = []

    if score >= 80:
        tips.append("🏆 <strong>Excellent match!</strong> Mirror the JD's exact phrasing in your bullet points to maximise ATS pass-through.")
    elif score >= 60:
        tips.append("👍 <strong>Good match.</strong> A few targeted additions could push your score above 80. Add the missing skills you genuinely possess.")
    elif score >= 40:
        tips.append("🤔 <strong>Moderate match.</strong> Consider a focused resume revision — use the JD's exact terminology throughout.")
    else:
        tips.append("⚠️ <strong>Low match.</strong> This role may require skills you haven't listed. Consider upskilling or targeting better-aligned roles.")

    if missing:
        tips.append(f"📌 <strong>Add these {len(missing)} skill(s)</strong> if you have them: "
                    + ", ".join(f"<code>{s}</code>" for s in missing[:15])
                    + ("…" if len(missing) > 15 else ""))
    else:
        tips.append("🎯 You've covered <em>all</em> skills mentioned in the JD — great!")

    tips.append("📝 Use the <strong>exact keywords</strong> from the JD — many ATS engines match verbatim phrases, not semantics.")
    tips.append("📊 <strong>Quantify achievements</strong>: 'Reduced latency by 35%' beats 'improved performance'.")
    tips.append("📄 Keep resume to <strong>1–2 pages</strong> with clear headings: Summary · Skills · Experience · Education · Projects.")
    if result["skill_score"] < result["keyword_score"]:
        tips.append("💡 Your general vocabulary overlaps well. Add a dedicated <strong>Technical Skills</strong> section listing tools explicitly.")

    return tips


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 6 — PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

def analyze_resume(filename: str, file_bytes: bytes, job_description: str) -> dict:
    """End-to-end pipeline returning a results dict."""
    resume_raw = extract_text(filename, file_bytes)
    if not resume_raw.strip():
        raise ValueError("Could not extract text from resume. Is the PDF image-only / scanned?")
    if not job_description.strip():
        raise ValueError("Job description is empty.")

    resume_tokens = preprocess_text(resume_raw)
    jd_tokens     = preprocess_text(job_description)
    resume_skills = extract_skills(resume_raw)
    jd_skills     = extract_skills(job_description)

    result = compute_ats_score(resume_skills, jd_skills, resume_tokens, jd_tokens)
    result["suggestions"]    = generate_suggestions(result)
    result["resume_word_ct"] = len(resume_raw.split())
    result["jd_word_ct"]     = len(job_description.split())
    return result


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 7 — HTML REPORT RENDERER
# ══════════════════════════════════════════════════════════════════════════════

def _score_color(score: float) -> str:
    if score >= 75: return "#34D399"   # green
    if score >= 50: return "#FBBF24"   # amber
    return "#F87171"                   # red

def _score_label(score: float) -> str:
    if score >= 80: return "Excellent match 🏆"
    if score >= 60: return "Good match 👍"
    if score >= 40: return "Moderate match 🤔"
    return "Low match ⚠️"

def build_html_report(r: dict, filename: str) -> str:
    score = r["ats_score"]
    col   = _score_color(score)
    label = _score_label(score)

    # ── Skill pill helpers ────────────────────────────────────────────────────
    def pills(items, bg, fg="#fff"):
        if not items:
            return '<span style="color:#718096;font-style:italic">None found</span>'
        return " ".join(
            f'<span style="display:inline-block;background:{bg};color:{fg};'
            f'border-radius:20px;padding:3px 10px;margin:3px;font-size:13px">{s}</span>'
            for s in items
        )

    matched_pills = pills(r["matched_skills"], "#1a4731", fg="#34D399")
    missing_pills = pills(r["missing_skills"], "#4a1919", fg="#F87171")
    extra_pills   = pills(r["extra_skills"],   "#1a2e4a", fg="#60A5FA")

    suggestions_html = "".join(
        f'<li style="margin-bottom:10px">{s}</li>' for s in r["suggestions"]
    )

    # ── SVG score ring ────────────────────────────────────────────────────────
    pct   = score / 100
    circ  = 2 * 3.14159 * 40          # circumference for r=40
    dash  = round(pct * circ, 2)
    gap   = round(circ - dash, 2)

    ring_svg = f"""
    <svg width="120" height="120" viewBox="0 0 120 120">
      <circle cx="60" cy="60" r="40" fill="none" stroke="#2E3350" stroke-width="10"/>
      <circle cx="60" cy="60" r="40" fill="none" stroke="{col}" stroke-width="10"
              stroke-dasharray="{dash} {gap}"
              stroke-dashoffset="{circ/4}"
              stroke-linecap="round" transform="rotate(-90 60 60)"/>
      <text x="60" y="65" text-anchor="middle" fill="{col}"
            font-size="20" font-weight="bold" font-family="monospace">{score}%</text>
    </svg>"""

    # ── Full HTML ─────────────────────────────────────────────────────────────
    html = f"""
<style>
  .ats-root {{
    font-family: 'Segoe UI', system-ui, sans-serif;
    background: #0F1117;
    color: #E2E8F0;
    border-radius: 12px;
    padding: 0 0 24px 0;
    max-width: 900px;
    margin: 12px auto;
    box-shadow: 0 8px 40px rgba(0,0,0,.5);
  }}
  .ats-header {{
    background: #1A1D27;
    border-radius: 12px 12px 0 0;
    padding: 18px 28px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    border-bottom: 1px solid #2E3350;
  }}
  .ats-header h2 {{ margin:0; font-size:20px; color:#A78BFA; }}
  .ats-header span {{ font-size:12px; color:#718096; }}
  .score-banner {{
    background: #1E293B;
    margin: 20px 24px 0;
    border-radius: 10px;
    padding: 18px 24px;
    display: flex;
    align-items: center;
    gap: 28px;
    flex-wrap: wrap;
  }}
  .sub-scores {{ margin-left: auto; text-align:right; }}
  .sub-scores p {{ margin:4px 0; font-size:14px; color:#94A3B8; }}
  .sub-scores strong {{ font-size:16px; }}
  .section {{
    margin: 20px 24px 0;
    background: #1A1D27;
    border-radius: 10px;
    padding: 16px 20px;
    border: 1px solid #2E3350;
  }}
  .section h3 {{
    margin: 0 0 12px 0;
    font-size: 14px;
    text-transform: uppercase;
    letter-spacing: .08em;
    color: #718096;
    border-bottom: 1px solid #2E3350;
    padding-bottom: 8px;
  }}
  .meta-grid {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(150px, 1fr));
    gap: 10px;
    margin-top: 8px;
  }}
  .meta-card {{
    background: #0F1117;
    border-radius: 8px;
    padding: 10px 14px;
    text-align: center;
  }}
  .meta-card .val {{ font-size:22px; font-weight:700; color:#4F8EF7; }}
  .meta-card .lbl {{ font-size:11px; color:#718096; margin-top:2px; }}
  .suggestions ol {{ padding-left: 20px; margin: 8px 0; }}
  .nlp-badge {{
    display:inline-block;
    background:#1a2e4a;
    color:#60A5FA;
    font-size:11px;
    border-radius:20px;
    padding:2px 10px;
    margin-left:8px;
  }}
</style>

<div class="ats-root">
  <!-- Header -->
  <div class="ats-header">
    <h2>⚡ ATS Resume Analyzer</h2>
    <span>📄 {filename} &nbsp;|&nbsp;
          NLP: {'NLTK ✓' if HAS_NLTK else 'Basic mode'}
          <span class="nlp-badge">{'pdfplumber' if HAS_PDFPLUMBER else 'PyPDF2' if HAS_PYPDF2 else '⚠ no PDF lib'}</span>
    </span>
  </div>

  <!-- Score banner -->
  <div class="score-banner">
    {ring_svg}
    <div>
      <div style="font-size:32px;font-weight:900;color:{col}">{score}%</div>
      <div style="color:#94A3B8;font-size:14px;margin-top:4px">{label}</div>
    </div>
    <div class="sub-scores">
      <p>Skill match score &nbsp;<strong style="color:#A78BFA">{r['skill_score']}%</strong></p>
      <p>Keyword overlap &nbsp;&nbsp;<strong style="color:#FBBF24">{r['keyword_score']}%</strong></p>
      <p style="margin-top:10px;font-size:12px">
        {r['common_tokens']} / {r['total_jd_tokens']} JD tokens matched
      </p>
    </div>
  </div>

  <!-- Stats row -->
  <div class="section">
    <h3>📈 Quick Stats</h3>
    <div class="meta-grid">
      <div class="meta-card">
        <div class="val">{len(r['matched_skills'])}</div>
        <div class="lbl">Matched Skills</div>
      </div>
      <div class="meta-card">
        <div class="val" style="color:#F87171">{len(r['missing_skills'])}</div>
        <div class="lbl">Missing Skills</div>
      </div>
      <div class="meta-card">
        <div class="val" style="color:#60A5FA">{len(r['extra_skills'])}</div>
        <div class="lbl">Extra Skills</div>
      </div>
      <div class="meta-card">
        <div class="val">{r['resume_word_ct']}</div>
        <div class="lbl">Resume Words</div>
      </div>
      <div class="meta-card">
        <div class="val">{r['jd_word_ct']}</div>
        <div class="lbl">JD Words</div>
      </div>
    </div>
  </div>

  <!-- Matched Skills -->
  <div class="section">
    <h3>✅ Matched Skills ({len(r['matched_skills'])})</h3>
    {matched_pills}
  </div>

  <!-- Missing Skills -->
  <div class="section">
    <h3>❌ Missing Skills ({len(r['missing_skills'])})</h3>
    {missing_pills}
  </div>

  <!-- Extra Skills -->
  <div class="section">
    <h3>➕ Additional Skills on Resume ({len(r['extra_skills'])})</h3>
    {extra_pills}
  </div>

  <!-- Suggestions -->
  <div class="section suggestions">
    <h3>💡 Improvement Suggestions</h3>
    <ol>{suggestions_html}</ol>
  </div>
</div>
"""
    return html


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 8 — COLAB UI  (ipywidgets)
# ══════════════════════════════════════════════════════════════════════════════

def launch_ats_app():
    """Build and display the interactive ATS widget in Colab."""

    # ── Dependency banner ─────────────────────────────────────────────────────
    dep_parts = []
    dep_parts.append("pdfplumber ✅" if HAS_PDFPLUMBER else ("PyPDF2 ✅" if HAS_PYPDF2 else "⚠️ No PDF lib — run: !pip install pdfplumber"))
    dep_parts.append("python-docx ✅" if HAS_DOCX else "⚠️ No DOCX lib — run: !pip install python-docx")
    dep_parts.append("NLTK ✅" if HAS_NLTK else "NLTK ✗ (basic mode — run: !pip install nltk)")

    display(HTML(f"""
    <div style="font-family:monospace;background:#1A1D27;color:#94A3B8;
                border-radius:8px;padding:10px 18px;margin-bottom:8px;font-size:12px">
      <strong style="color:#A78BFA">⚡ ATS Resume Analyzer — Colab Edition</strong>
      &nbsp;&nbsp;|&nbsp;&nbsp;
      {"&nbsp; | &nbsp;".join(dep_parts)}
    </div>
    """))

    # ── Widgets ───────────────────────────────────────────────────────────────
    upload_btn = widgets.FileUpload(
        accept=".pdf,.docx,.doc",
        multiple=False,
        description="📂 Upload Resume",
        layout=widgets.Layout(width="220px"),
        style={"button_color": "#4F8EF7", "button_color_active": "#3B7AE0"},
    )

    file_info = widgets.HTML(
        value='<span style="color:#718096;font-size:13px">No file selected yet.</span>'
    )

    jd_box = widgets.Textarea(
        placeholder="Paste the full job description here…",
        layout=widgets.Layout(width="100%", height="200px"),
    )

    analyze_btn = widgets.Button(
        description="🔍  Analyze Resume",
        button_style="",
        layout=widgets.Layout(width="200px", height="40px"),
        style={"button_color": "#4F8EF7", "font_weight": "bold"},
    )

    clear_btn = widgets.Button(
        description="🗑 Clear",
        button_style="",
        layout=widgets.Layout(width="100px", height="40px"),
        style={"button_color": "#374151"},
    )

    status_bar = widgets.HTML(
        value='<span style="color:#718096;font-size:12px">Ready.</span>'
    )

    output_area = widgets.Output()

    # ── Layout ────────────────────────────────────────────────────────────────
    display(widgets.VBox([
        widgets.HBox([upload_btn, file_info],
                     layout=widgets.Layout(align_items="center", gap="14px")),
        widgets.HTML('<b style="color:#A78BFA;font-size:13px">💼 Job Description</b>'),
        jd_box,
        widgets.HBox([analyze_btn, clear_btn, status_bar],
                     layout=widgets.Layout(align_items="center", gap="12px")),
        output_area,
    ], layout=widgets.Layout(gap="10px", padding="4px")))

    # ── Callbacks ─────────────────────────────────────────────────────────────

    def on_upload_change(change):
        if upload_btn.value:
            fname = list(upload_btn.value.keys())[0]
            size  = len(list(upload_btn.value.values())[0]["content"]) // 1024
            file_info.value = (
                f'<span style="color:#34D399;font-size:13px">'
                f'📄 {fname} &nbsp;({size} KB)</span>'
            )

    upload_btn.observe(on_upload_change, names="value")

    def on_analyze(b):
        # Validate inputs
        if not upload_btn.value:
            status_bar.value = '<span style="color:#F87171">⚠ Please upload a resume first.</span>'
            return
        jd_text = jd_box.value.strip()
        if not jd_text:
            status_bar.value = '<span style="color:#F87171">⚠ Please enter a job description.</span>'
            return

        analyze_btn.disabled = True
        status_bar.value = '<span style="color:#FBBF24">⏳ Analyzing…</span>'

        try:
            fname     = list(upload_btn.value.keys())[0]
            fbytes    = bytes(list(upload_btn.value.values())[0]["content"])
            result    = analyze_resume(fname, fbytes, jd_text)
            html_rep  = build_html_report(result, fname)

            with output_area:
                clear_output(wait=True)
                display(HTML(html_rep))

            status_bar.value = (
                f'<span style="color:#34D399">✅ Done — ATS Score: '
                f'<strong>{result["ats_score"]}%</strong> &nbsp;|&nbsp; '
                f'{len(result["matched_skills"])} matched, '
                f'{len(result["missing_skills"])} missing skills.</span>'
            )

        except Exception as exc:
            with output_area:
                clear_output(wait=True)
                display(HTML(
                    f'<div style="background:#4a1919;color:#F87171;border-radius:8px;'
                    f'padding:14px 20px;font-family:monospace">'
                    f'<strong>Error:</strong> {exc}</div>'
                ))
            status_bar.value = f'<span style="color:#F87171">❌ {exc}</span>'

        finally:
            analyze_btn.disabled = False

    def on_clear(b):
        with output_area:
            clear_output()
        jd_box.value    = ""
        status_bar.value = '<span style="color:#718096;font-size:12px">Cleared.</span>'
        file_info.value  = '<span style="color:#718096;font-size:13px">No file selected yet.</span>'

    analyze_btn.on_click(on_analyze)
    clear_btn.on_click(on_clear)


# ══════════════════════════════════════════════════════════════════════════════
# LAUNCH
# ══════════════════════════════════════════════════════════════════════════════
launch_ats_app()
